# Dynamic Image vs. Multimodal Significance Tests

This notebook implements the statistical comparison between the dynamic image-based and multimodal FER models.

The test set contains **378 reenactments** with two view-level samples per reenactment:

- one Central-view sample
- one Side-view sample

Both models therefore produce predictions for the same **756 view-level samples**. The pooled accuracy difference is computed over all 756 predictions, while statistical resampling uses the 378 reenactments as paired clusters so that Central and Side from the same reenactment are not treated as independent observations.

## Analysis plan

1. Primary comparison: compare pooled Multimodal and Image accuracy using a centered paired cluster bootstrap over reenactments.
2. Sensitivity check: test the same reenactment-level mean difference with a one-sample t-test.
3. Participant-level heterogeneity check: summarize the pooled Multimodal-vs.-Image difference separately for each of the eight participants.

### Note on the primary resampling test

Both Image and Multimodal correctness have the same view-level support $\{0,1\}$ and the same reenactment-level support $\{0,0.5,1\}$. A paired label-permutation/sign-flip test would therefore be feasible.

For consistency with the other modality comparisons and to target the pooled accuracy difference directly without assuming independence of all 756 view samples, the primary significance test uses a **centered paired cluster bootstrap** over reenactments. The same resampling scheme is used to obtain the 95% confidence interval.


## 1. Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.stats import ttest_1samp


PREDICTIONS_PATH = Path("dynamic_test_predictions.csv")

RANDOM_SEED = 42

# Use many resamples for stable final confidence intervals and p-values.
# Resampling is processed in batches below to keep memory usage low.
N_BOOTSTRAP = 1_000_000
BOOTSTRAP_BATCH_SIZE = 10_000

ALPHA = 0.05


## 2. Load and Validate Predictions

In [2]:
prediction_df = pd.read_csv(PREDICTIONS_PATH)

print(f"Rows: {len(prediction_df)}")
print(f"Columns: {len(prediction_df.columns)}")
prediction_df.head()


Rows: 756
Columns: 39


,sample_id,reenactment_id,timestamp,set_id,participant_id,level_id,emoji_id,camera_index,perspective,true_label_id,...,fea_prob_surprise,multimodal_pred_id,multimodal_pred,multimodal_prob_anger,multimodal_prob_disgust,multimodal_prob_fear,multimodal_prob_happiness,multimodal_prob_neutral,multimodal_prob_sadness,multimodal_prob_surprise
0,1700478995850-2-1-1-0-0-0,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,0,Central,0,...,0.016200,0,Anger,0.798008,0.054746,0.005280,0.006042,0.009937,0.117664,0.008323
1,1700478995850-2-1-1-0-0-1,1700478995850-2-1-1-0-0,1700478995850,2,1,1,0,1,Side,0,...,0.016200,0,Anger,0.586113,0.127385,0.010630,0.006146,0.022720,0.239457,0.007548
2,1700478998549-2-1-1-1-5-0,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,0,Central,5,...,0.000089,5,Sadness,0.016788,0.003464,0.003549,0.006522,0.007196,0.958934,0.003547
3,1700478998549-2-1-1-1-5-1,1700478998549-2-1-1-1-5,1700478998549,2,1,1,1,1,Side,5,...,0.000089,5,Sadness,0.016804,0.003480,0.003540,0.006520,0.007197,0.958908,0.003550
4,1700479001137-2-1-1-2-3-0,1700479001137-2-1-1-2-3,1700479001137,2,1,1,2,0,Central,3,...,0.000331,3,Happiness,0.002462,0.004623,0.003282,0.982724,0.001283,0.002237,0.003387


In [3]:
required_columns = {
    "sample_id",
    "reenactment_id",
    "participant_id",
    "camera_index",
    "true_label_id",
    "image_pred_id",
    "multimodal_pred_id"
}

missing_columns = required_columns - set(prediction_df.columns)
assert not missing_columns, f"Missing required columns: {sorted(missing_columns)}"

assert len(prediction_df) == 756
assert prediction_df["sample_id"].is_unique
assert prediction_df["reenactment_id"].nunique() == 378
assert set(prediction_df["camera_index"].unique()) == {0, 1}

samples_per_reenactment = prediction_df.groupby("reenactment_id").size()
assert samples_per_reenactment.eq(2).all()

views_per_reenactment = prediction_df.groupby("reenactment_id")["camera_index"].nunique()
assert views_per_reenactment.eq(2).all()

true_labels_per_reenactment = prediction_df.groupby("reenactment_id")["true_label_id"].nunique()
assert true_labels_per_reenactment.eq(1).all()

participants_per_reenactment = prediction_df.groupby("reenactment_id")["participant_id"].nunique()
assert participants_per_reenactment.eq(1).all()
assert prediction_df["participant_id"].nunique() == 8

print("Prediction-table structure validated.")


Prediction-table structure validated.


## 3. Construct Reenactment-Level Analysis Table

In [4]:
prediction_df = prediction_df.copy()

prediction_df["image_correct"] = prediction_df["image_pred_id"] == prediction_df["true_label_id"]
prediction_df["multimodal_correct"] = prediction_df["multimodal_pred_id"] == prediction_df["true_label_id"]


In [5]:
image_correct_by_view = (
    prediction_df
    .pivot(index="reenactment_id", columns="camera_index", values="image_correct")
    .rename(columns={
        0: "central_image_correct",
        1: "side_image_correct"
    })
)

multimodal_correct_by_view = (
    prediction_df
    .pivot(index="reenactment_id", columns="camera_index", values="multimodal_correct")
    .rename(columns={
        0: "central_multimodal_correct",
        1: "side_multimodal_correct"
    })
)

participant_id = prediction_df.groupby("reenactment_id")["participant_id"].first()

analysis_df = image_correct_by_view.join(multimodal_correct_by_view).join(participant_id)

correctness_columns = [
    "central_image_correct",
    "side_image_correct",
    "central_multimodal_correct",
    "side_multimodal_correct"
]

analysis_df[correctness_columns] = analysis_df[correctness_columns].astype(bool)

analysis_df["image_correct_mean"] = (
    analysis_df["central_image_correct"].astype(float)
    + analysis_df["side_image_correct"].astype(float)
) / 2.0

analysis_df["multimodal_correct_mean"] = (
    analysis_df["central_multimodal_correct"].astype(float)
    + analysis_df["side_multimodal_correct"].astype(float)
) / 2.0

assert len(analysis_df) == 378
assert not analysis_df.isna().any().any()
assert set(analysis_df["image_correct_mean"].unique()).issubset({0.0, 0.5, 1.0})
assert set(analysis_df["multimodal_correct_mean"].unique()).issubset({0.0, 0.5, 1.0})

analysis_df.head()


,central_image_correct,side_image_correct,central_multimodal_correct,side_multimodal_correct,participant_id,image_correct_mean,multimodal_correct_mean
reenactment_id,,,,,,,
1700478995850-2-1-1-0-0,True,False,True,True,1,0.5,1.0
1700478998549-2-1-1-1-5,True,True,True,True,1,1.0,1.0
1700479001137-2-1-1-2-3,True,False,True,True,1,0.5,1.0
1700479004312-2-1-1-3-0,False,False,False,False,1,0.0,0.0
1700479005401-2-1-1-4-0,True,True,True,True,1,1.0,1.0


## 4. Verify Reported Performance

In [6]:
pooled_image_accuracy = analysis_df["image_correct_mean"].mean()
pooled_multimodal_accuracy = analysis_df["multimodal_correct_mean"].mean()

central_image_accuracy = analysis_df["central_image_correct"].mean()
side_image_accuracy = analysis_df["side_image_correct"].mean()
central_multimodal_accuracy = analysis_df["central_multimodal_correct"].mean()
side_multimodal_accuracy = analysis_df["side_multimodal_correct"].mean()

accuracy_difference = pooled_multimodal_accuracy - pooled_image_accuracy

print(f"Pooled image accuracy:           {pooled_image_accuracy:.4%}")
print(f"Pooled multimodal accuracy:      {pooled_multimodal_accuracy:.4%}")
print(f"Central image accuracy:          {central_image_accuracy:.4%}")
print(f"Side image accuracy:             {side_image_accuracy:.4%}")
print(f"Central multimodal accuracy:     {central_multimodal_accuracy:.4%}")
print(f"Side multimodal accuracy:        {side_multimodal_accuracy:.4%}")
print(f"Multimodal - Image:              {100 * accuracy_difference:.2f} percentage points")

assert np.isclose(pooled_image_accuracy, prediction_df["image_correct"].mean())
assert np.isclose(pooled_multimodal_accuracy, prediction_df["multimodal_correct"].mean())

assert prediction_df["image_correct"].sum() == 550
assert np.isclose(pooled_image_accuracy, 550 / 756)

assert prediction_df["multimodal_correct"].sum() == 617
assert np.isclose(pooled_multimodal_accuracy, 617 / 756)


Pooled image accuracy:           72.7513%
Pooled multimodal accuracy:      81.6138%
Central image accuracy:          73.0159%
Side image accuracy:             72.4868%
Central multimodal accuracy:     81.7460%
Side multimodal accuracy:        81.4815%
Multimodal - Image:              8.86 percentage points


## 5. Primary Image vs. Multimodal Comparison

The primary estimand is the difference between the reported pooled accuracies, $\Delta = \mathrm{Accuracy}_{Multimodal} - \mathrm{Accuracy}_{Image}$.

For each reenactment, the two view-level paired differences are averaged:

$d_i = \big[(M_{i,C} - I_{i,C}) + (M_{i,S} - I_{i,S})\big] / 2$

The mean of these 378 reenactment-level differences is exactly the difference between the two pooled accuracies over all 756 view samples.

The 378 reenactment-level differences are used as the resampling units.

The analysis estimates:

1. a two-sided bootstrap p-value for $H_0: \Delta = 0$, using the centered empirical distribution under the null
2. a percentile-bootstrap 95% confidence interval for $\Delta$

Positive values favor Multimodal.


In [7]:
def paired_cluster_bootstrap(differences: np.ndarray,
                             n_bootstrap: int = N_BOOTSTRAP,
                             batch_size: int = BOOTSTRAP_BATCH_SIZE,
                             seed: int = RANDOM_SEED,
                             alpha: float = ALPHA) -> dict:
    
    differences = np.asarray(differences, dtype=float)

    if differences.ndim != 1 or differences.size == 0:
        raise ValueError("differences must be a nonempty one-dimensional array.")
    if not np.isin(differences, [-1.0, -0.5, 0.0, 0.5, 1.0]).all():
        raise ValueError("Expected paired accuracy differences in {-1, -0.5, 0, 0.5, 1}.")

    n = len(differences)
    observed_difference = differences.mean()

    doubled_differences = (2 * differences).astype(np.int64)
    observed_sum = doubled_differences.sum()

    rng = np.random.default_rng(seed)
    bootstrap_means = np.empty(n_bootstrap, dtype=float)
    n_extreme = 0

    for start in range(0, n_bootstrap, batch_size):
        end = min(start + batch_size, n_bootstrap)
        current_batch_size = end - start

        # Ordinary paired bootstrap for the confidence interval.
        bootstrap_indices = rng.integers(0, n, size=(current_batch_size, n))
        bootstrap_means[start:end] = differences[bootstrap_indices].mean(axis=1)

        # Centered-bootstrap null test evaluated in exact integer arithmetic:
        # |mean(d*) - mean(d)| >= |mean(d)| is equivalent to |S* - S| >= |S|,
        # where q_i = 2 d_i, S = sum(q_i), and S* = sum(q_i*).
        null_indices = rng.integers(0, n, size=(current_batch_size, n))
        null_sums = doubled_differences[null_indices].sum(axis=1)

        n_extreme += np.count_nonzero(
            np.abs(null_sums - observed_sum) >= abs(observed_sum)
        )

    ci_low, ci_high = np.quantile(bootstrap_means, [alpha / 2, 1 - alpha / 2])
    p_value = (n_extreme + 1) / (n_bootstrap + 1)

    return {
        "n": n,
        "difference": observed_difference,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": p_value,
        "n_bootstrap": n_bootstrap
    }


In [8]:
reenactment_differences = (
    analysis_df["multimodal_correct_mean"] - analysis_df["image_correct_mean"]
).to_numpy()

assert np.isclose(reenactment_differences.mean(), accuracy_difference)

primary_result = paired_cluster_bootstrap(
    differences=reenactment_differences,
    n_bootstrap=N_BOOTSTRAP,
    batch_size=BOOTSTRAP_BATCH_SIZE,
    seed=RANDOM_SEED,
    alpha=ALPHA
)

primary_result_df = pd.DataFrame([{
    "comparison": "Pooled multimodal - pooled image",
    "n_reenactments": primary_result["n"],
    "image_accuracy": pooled_image_accuracy,
    "multimodal_accuracy": pooled_multimodal_accuracy,
    "difference_pp": 100 * primary_result["difference"],
    "ci_low_pp": 100 * primary_result["ci_low"],
    "ci_high_pp": 100 * primary_result["ci_high"],
    "p_value": primary_result["p_value"],
    "n_bootstrap": primary_result["n_bootstrap"]
}])

primary_result_df


,comparison,n_reenactments,image_accuracy,multimodal_accuracy,difference_pp,ci_low_pp,ci_high_pp,p_value,n_bootstrap
0,Pooled multimodal - pooled image,378,0.727513,0.816138,8.862434,5.687831,12.037037,9.999990e-07,1000000


In [9]:
row = primary_result_df.iloc[0]

print(f"Multimodal - pooled image accuracy difference: {row['difference_pp']:.2f} percentage points")
print(f"{100 * (1 - ALPHA):.0f}% bootstrap CI: [{row['ci_low_pp']:.2f}, {row['ci_high_pp']:.2f}] percentage points")
print(f"Two-sided bootstrap p-value: {row['p_value']:.12f}")


Multimodal - pooled image accuracy difference: 8.86 percentage points
95% bootstrap CI: [5.69, 12.04] percentage points
Two-sided bootstrap p-value: 0.000000999999


## 6. Sensitivity Check

As a sensitivity analysis, apply a one-sample t-test to the same 378 reenactment-level paired differences.

This is not a separate research hypothesis and is not included in a multiplicity-correction family. It checks whether the inferential conclusion agrees with the primary bootstrap analysis.


In [10]:
sensitivity_test = ttest_1samp(reenactment_differences, popmean=0)

sensitivity_result_df = pd.DataFrame([{
    "comparison": "Pooled multimodal - pooled image",
    "n_reenactments": len(reenactment_differences),
    "difference_pp": 100 * reenactment_differences.mean(),
    "t_statistic": sensitivity_test.statistic,
    "degrees_of_freedom": sensitivity_test.df,
    "p_value": sensitivity_test.pvalue
}])

sensitivity_result_df


,comparison,n_reenactments,difference_pp,t_statistic,degrees_of_freedom,p_value
0,Pooled multimodal - pooled image,378,8.862434,5.407263,377,1.138446e-07


In [11]:
row = sensitivity_result_df.iloc[0]

print(f"Mean difference: {row['difference_pp']:.2f} percentage points")
print(f"t({row['degrees_of_freedom']:.0f}) = {row['t_statistic']:.3f}")
print(f"Two-sided sensitivity-check p-value: {row['p_value']:.12f}")


Mean difference: 8.86 percentage points
t(377) = 5.407
Two-sided sensitivity-check p-value: 0.000000113845


## 7. Participant-Level Heterogeneity Check

This descriptive check examines whether the pooled Multimodal-vs.-Image difference is directionally consistent across the eight test participants or is mainly driven by a small number of participants.

For each participant, the table reports the number of reenactments, pooled Image accuracy, pooled Multimodal accuracy, and the difference $\mathrm{Accuracy}_{Multimodal} - \mathrm{Accuracy}_{Image}$.

No participant-level significance test or correction factor is applied. The primary inference remains the reenactment-level analysis above; this section is a heterogeneity and plausibility check.


In [12]:
participant_result_df = (
    analysis_df.reset_index()
    .groupby("participant_id", as_index=False)
    .agg(
        n_reenactments=("reenactment_id", "size"),
        image_accuracy=("image_correct_mean", "mean"),
        multimodal_accuracy=("multimodal_correct_mean", "mean")
    )
)

participant_result_df["difference_pp"] = 100 * (
    participant_result_df["multimodal_accuracy"] - participant_result_df["image_accuracy"]
)

n_multimodal_better = int((participant_result_df["difference_pp"] > 0).sum())
n_image_better = int((participant_result_df["difference_pp"] < 0).sum())
n_equal = int((participant_result_df["difference_pp"] == 0).sum())
median_participant_difference_pp = participant_result_df["difference_pp"].median()

print(f"Participants favoring Multimodal: {n_multimodal_better}/8")
print(f"Participants favoring Image:      {n_image_better}/8")
print(f"Participants tied:                 {n_equal}/8")
print(f"Median participant difference (Multimodal - Image): {median_participant_difference_pp:.2f} percentage points")

participant_result_df


Participants favoring Multimodal: 8/8
Participants favoring Image:      0/8
Participants tied:                 0/8
Median participant difference (Multimodal - Image): 8.59 percentage points


,participant_id,n_reenactments,image_accuracy,multimodal_accuracy,difference_pp
0,1,47,0.829787,0.861702,3.191489
1,8,54,0.555556,0.703704,14.814815
2,10,46,0.619565,0.728261,10.869565
3,13,46,0.804348,0.891304,8.695652
4,15,48,0.718750,0.781250,6.250000
5,18,43,0.860465,0.965116,10.465116
6,23,53,0.773585,0.858491,8.490566
7,27,41,0.682927,0.756098,7.317073


## 8. Summary and Export

The primary result compares the reported pooled dynamic Multimodal and Image accuracies while preserving the dependency between Central and Side samples from the same reenactment. The one-sample t-test provides a sensitivity check of the same mean difference. The participant-level breakdown is descriptive and is used to assess directional consistency and heterogeneity across the eight test participants.


In [13]:
summary_df = pd.DataFrame({
    "metric": [
        "Pooled image accuracy",
        "Pooled multimodal accuracy",
        "Multimodal - Image difference (pp)",
        "Primary bootstrap CI low (pp)",
        "Primary bootstrap CI high (pp)",
        "Primary bootstrap p-value",
        "Sensitivity t statistic",
        "Sensitivity t-test p-value",
        "Participants favoring Multimodal",
        "Participants favoring Image",
        "Participants tied",
        "Median participant difference Multimodal - Image (pp)"
    ],
    "value": [
        pooled_image_accuracy,
        pooled_multimodal_accuracy,
        100 * primary_result["difference"],
        100 * primary_result["ci_low"],
        100 * primary_result["ci_high"],
        primary_result["p_value"],
        sensitivity_test.statistic,
        sensitivity_test.pvalue,
        n_multimodal_better,
        n_image_better,
        n_equal,
        median_participant_difference_pp
    ]
})

summary_df


,metric,value
0,Pooled image accuracy,7.275132e-01
1,Pooled multimodal accuracy,8.161376e-01
2,Multimodal - Image difference (pp),8.862434e+00
3,Primary bootstrap CI low (pp),5.687831e+00
4,Primary bootstrap CI high (pp),1.203704e+01
5,Primary bootstrap p-value,9.999990e-07
6,Sensitivity t statistic,5.407263e+00
7,Sensitivity t-test p-value,1.138446e-07
8,Participants favoring Multimodal,8.000000e+00
9,Participants favoring Image,0.000000e+00


In [14]:
OUTPUT_DIR = Path("statistical_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

analysis_df.to_csv(OUTPUT_DIR / "dynamic_image_vs_multimodal_reenactment_table.csv", index=True)
primary_result_df.to_csv(OUTPUT_DIR / "dynamic_image_vs_multimodal_primary_result.csv", index=False)
sensitivity_result_df.to_csv(OUTPUT_DIR / "dynamic_image_vs_multimodal_sensitivity_ttest.csv", index=False)
participant_result_df.to_csv(OUTPUT_DIR / "dynamic_image_vs_multimodal_participant_level_results.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "dynamic_image_vs_multimodal_summary.csv", index=False)

print(f"Results written to: {OUTPUT_DIR.resolve()}")


Results written to: /workspace/repos/emohevrdb-dfer/6_discussion/significance-tests/dynamic-significance-tests/statistical_results
